In [1]:
# 필수 실행 - 1
from google.colab import drive
import os

# 1. 드라이브 마운트
drive.mount('/content/drive')

# 2. 작업 경로 설정 및 이동
base_path = "/content/drive/MyDrive/Mecro" # 태영 님의 폴더 경로
if not os.path.exists(base_path):
    os.makedirs(base_path)

%cd {base_path}
print(f"현재 작업 경로: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Mecro
현재 작업 경로: /content/drive/MyDrive/Mecro


In [2]:
# [수정 사항] 의존성 충돌 방지 및 최신 라이브러리 설치 - 필수 실행
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-http
!pip install -q -U langchain-google-genai langchain-chroma chromadb langchain_community opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-http

Found existing installation: opentelemetry-api 1.41.0
Uninstalling opentelemetry-api-1.41.0:
  Successfully uninstalled opentelemetry-api-1.41.0
Found existing installation: opentelemetry-sdk 1.41.0
Uninstalling opentelemetry-sdk-1.41.0:
  Successfully uninstalled opentelemetry-sdk-1.41.0
Found existing installation: opentelemetry-exporter-otlp-proto-http 1.41.0
Uninstalling opentelemetry-exporter-otlp-proto-http-1.41.0:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-http-1.41.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1

In [3]:
# 필요한 라이브러리 설치 & 필수 실행 - 2
!pip install yfinance beautifulsoup4 pandas requests

import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import os

# 데이터 저장 경로 설정
DATA_PATH = "./data/raw"
if not os.path.exists(DATA_PATH):
  os.makedirs(DATA_PATH)

In [4]:
# 필수 실행 - 3
!pip install feedparser

In [5]:
# [Section 01] 데이터 팩토리 최적화 가동 - 필수 실행
import feedparser
import pandas as pd
import yfinance as yf
from datetime import datetime
import time
import os

# 1. 전역 설정
DATA_PATH = "/content/drive/MyDrive/Mecro/data"
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

RSS_SOURCES = {
    "Yahoo_Main": "https://finance.yahoo.com/news/rss",
    "Yahoo_CentralBank": "https://finance.yahoo.com/news/category-central-banks/rss",
    "Yahoo_Economy": "https://finance.yahoo.com/news/category-economy/rss"
}

# [추가] 거시경제 필터링 함수: 개인 금융 및 광고성 노이즈 뉴스 제거
def is_macro_news(title):
    # 가독성을 저해하는 개인 금융 키워드 목록
    exclude_keywords = [
        'CD rates', 'Credit score', 'Sallie Mae', 'Personal Finance',
        'Tax refund', 'Best banks', 'HELOC', 'Mortgage rates today',
        'Credit card', 'Savings account'
    ]
    # 제외 키워드가 제목에 포함되어 있으면 False 반환
    return not any(keyword.lower() in title.lower() for keyword in exclude_keywords)

# 2. 뉴스 수집 함수 (필터링 및 2026년 기간 제한 적용)
def fetch_accumulated_news(limit_year="2026"):
    all_news = []
    for name, url in RSS_SOURCES.items():
        print(f"📡 {name} 피드 분석 중...")
        feed = feedparser.parse(url)
        for entry in feed.entries:
            # 필터 1: 거시경제 관련 뉴스 여부 확인
            if not is_macro_news(entry.title):
                continue

            try:
                published_at = time.strftime('%Y-%m-%d %H:%M:%S', entry.published_parsed)
            except:
                published_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            # 필터 2: 2026년 데이터만 수집 (과거 데이터 배제)
            if not published_at.startswith(limit_year):
                continue

            all_news.append({
                "title": entry.title,
                "url": entry.link,
                "published_at": published_at,
                "source": name,
                "context_text": f"[{published_at}] {name}: {entry.title}"
            })

    file_path = f"{DATA_PATH}/raw_news.csv"

    if not all_news:
        print("💡 수집된 신규 거시경제 뉴스가 없습니다.")
        return pd.read_csv(file_path) if os.path.exists(file_path) else pd.DataFrame(), pd.DataFrame()

    new_df = pd.DataFrame(all_news)

    if os.path.exists(file_path):
        old_df = pd.read_csv(file_path)
        # 기존 데이터에서도 2026년 이전 기록은 제거하여 최신성 유지
        old_df = old_df[old_df['published_at'].str.startswith(limit_year)]
        new_only_df = new_df[~new_df['url'].isin(old_df['url'])].copy()
        final_df = pd.concat([old_df, new_only_df]).drop_duplicates(subset=['url'], keep='first')
    else:
        new_only_df = new_df.copy()
        final_df = new_df

    final_df.sort_values(by='published_at', ascending=False, inplace=True)
    final_df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"✅ 거시경제 뉴스 업데이트 완료: 총 {len(final_df)}개 (신규: {len(new_only_df)}건)")
    return final_df, new_only_df

# 3. 마켓 지수 수집 함수 (은(Silver) 포함 및 시간 단위 최적화)
def fetch_robust_market_data(period="1mo"):
    # 반도체 및 매크로 핵심 자산 구성
    tickers = {
        "10Y_Bond": "^TNX",
        "Gold": "GC=F",
        "Silver": "SI=F",      # 반도체 핵심 원자재 추가
        "Copper": "HG=F",
        "USD_Index": "DX-Y.NYB",
        "Aluminum": "ALI=F"
    }

    all_data = []
    for name, ticker in tickers.items():
        print(f"📈 {name} 로드 중...")
        try:
            df = yf.Ticker(ticker).history(period=period, interval="1h")
            if not df.empty:
                df = df[['Close']].rename(columns={'Close': name})
                # 시간대 보정 및 시간 단위 절삭
                df.index = df.index.tz_localize(None).floor('h')
                all_data.append(df)
        except Exception as e:
            print(f"⚠️ {name} 실패: {e}")

    if not all_data:
        return pd.DataFrame()

    market_df = pd.concat(all_data, axis=1)
    market_df = market_df[~market_df.index.duplicated(keep='first')]

    # 누락된 시간대 보간 처리
    full_index = pd.date_range(start=market_df.index.min(), end=market_df.index.max(), freq='h')
    market_df = market_df.reindex(full_index).ffill().bfill()

    market_df.to_csv(f"{DATA_PATH}/market_prices.csv")
    print(f"✅ 은(Silver) 포함 마켓 데이터 완료: {len(market_df)} 행")
    return market_df

# 4. 최종 실행
news_data, new_news_to_analyze = fetch_accumulated_news(limit_year="2026")
market_data = fetch_robust_market_data(period="1mo")

📡 Yahoo_Main 피드 분석 중...
📡 Yahoo_CentralBank 피드 분석 중...
📡 Yahoo_Economy 피드 분석 중...
✅ 거시경제 뉴스 업데이트 완료: 총 183개 (신규: 10건)
📈 10Y_Bond 로드 중...
📈 Gold 로드 중...
📈 Silver 로드 중...
📈 Copper 로드 중...
📈 USD_Index 로드 중...
📈 Aluminum 로드 중...
✅ 은(Silver) 포함 마켓 데이터 완료: 791 행


In [6]:
# 필수 실행 - 5
# 01번 노트북: 수집한 데이터를 파일로 '저장'하기
# 01_Data_Factory.ipynb의 마지막 부분
import pandas as pd

# 수집된 뉴스 데이터가 담긴 DataFrame이 'df'라고 가정합니다.
# 'index=False'를 넣어야 나중에 불러올 때 불필요한 번호 열이 생기지 않습니다.
news_data.to_csv('news_data_final.csv', index=False, encoding='utf-8-sig')

print("✅ 뉴스 데이터가 'news_data_final.csv' 파일로 저장되었습니다.")

✅ 뉴스 데이터가 'news_data_final.csv' 파일로 저장되었습니다.


In [7]:
# 필수 실행 - 6
# 해결 방법 1: 사용 가능한 모델 목록 확인 (Debugging)
import google.generativeai as genai
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# 현재 사용 가능한 모든 모델 출력
print("--- 사용 가능한 임베딩 모델 목록 ---")
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


--- 사용 가능한 임베딩 모델 목록 ---
models/gemini-embedding-001
models/gemini-embedding-2-preview


In [8]:
# 필수 실행 - 7
# 해결 방법: 사용 가능한 모델로 코드 수정
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata
import os

# API 키 설정 - 런타임 재시작 시 다시 실행 필요함
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# 확인된 모델 목록 중 표준 모델인 gemini-embedding-2-preview을 사용합니다.
# 'models/' 접두사를 포함하여 정확히 입력합니다.
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview",
    google_api_key=GOOGLE_API_KEY
)

# 실제 작동 확인을 위한 테스트 코드
try:
    test_vec = embeddings.embed_query("뉴스 데이터 임베딩 테스트를 진행합니다.")
    print(f"✅ 모델 로드 및 임베딩 성공! (벡터 차원: {len(test_vec)})")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

✅ 모델 로드 및 임베딩 성공! (벡터 차원: 3072)


In [9]:
# 필수 실행 - 9
try:
    from langchain_community.vectorstores import Chroma
    print("✅ langchain_community 및 ChromaDB 로드 성공")
except ImportError as e:
    print(f"❌ 여전히 로드 실패: {e}")

✅ langchain_community 및 ChromaDB 로드 성공


In [10]:
# [참고] new_documents를 생성하는 코드 예시
# 이 코드가 [Section 02] 실행 전에 반드시 완료되어야 합니다. - 필수 실행

from langchain_core.documents import Document

new_documents = []

# 업로드하신 csv 파일 구조(title, url, published_at, source, context_text)에 맞게 수정했습니다.
for _, row in new_news_to_analyze.iterrows():
    # 1. page_content: 분석의 핵심이 되는 내용 ('context_text' 열을 사용합니다)
    doc = Document(
        page_content=row['context_text'],
        metadata={
            "title": row['title'],
            "press": row['source'],        # 'press' 대신 'source' 사용
            "date": row['published_at'],   # 'date' 대신 'published_at' 사용
            "link": row['url']             # 'link' 대신 'url' 사용
        }
    )
    new_documents.append(doc)

print(f"✅ 총 {len(new_documents)}건의 뉴스 데이터를 실제 파일 구조에 맞춰 변환 완료했습니다!")

✅ 총 10건의 뉴스 데이터를 실제 파일 구조에 맞춰 변환 완료했습니다!


In [11]:
# [Section 02] 벡터 DB 누적 업데이트 (기존 데이터 보존 방식) - 필수 실행
import shutil
import os
import time
import chromadb
from chromadb.config import Settings
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from google.colab import userdata

# 1. API 키 및 임베딩 모델 설정
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")
except Exception as e:
    print(f"❌ 보안 비밀 로드 실패: {e}")

# 2. 경로 설정
local_db_path = "/content/temp_vector_db"
drive_db_path = "/content/drive/MyDrive/Mecro/vector_db_2026_v3"

# ✅ [수정 포인트 1] 초기화 대신 기존 데이터 불러오기
# 로컬 작업 경로가 이미 있다면 깨끗하게 정리 (현재 세션 작업용)
if os.path.exists(local_db_path):
    shutil.rmtree(local_db_path)

# 드라이브에 기존 DB가 있다면 로컬로 복사해서 가져옴 (누적의 핵심)
if os.path.exists(drive_db_path):
    print("📂 기존 벡터 DB를 드라이브에서 로컬로 로드합니다...")
    shutil.copytree(drive_db_path, local_db_path)
else:
    print("🆕 기존 DB가 없습니다. 새로운 데이터베이스를 생성합니다.")

# 3. ChromaDB 클라이언트 설정
try:
    client = chromadb.PersistentClient(
        path=local_db_path,
        settings=Settings(
            is_persistent=True,
            anonymized_telemetry=False,
            allow_reset=True
        )
    )

    vector_db = Chroma(
        client=client,
        embedding_function=embeddings,
        collection_name="macro_intelligence_db_2026"
    )
    print(f"✅ 벡터 DB 연결 성공 (현재 저장된 문서: {vector_db._collection.count()}개)")

except Exception as e:
    print(f"❌ 치명적 오류: {e}")

# 4. 신규 데이터 추가 및 드라이브 동기화
if 'new_news_to_analyze' in locals() or 'new_news_to_analyze' in globals():
    if not new_news_to_analyze.empty and 'new_documents' in locals():
        total_docs = len(new_news_to_analyze)
        print(f"🚀 신규 뉴스 {total_docs}건 추가 저장을 시작합니다...")

        # 기존 데이터가 담긴 로컬 DB에 새 문서 추가
        vector_db.add_documents(new_documents)

        # ✅ [수정 포인트 2] 업데이트된 로컬 DB를 다시 드라이브로 복사
        print("💾 업데이트된 데이터를 드라이브로 동기화 중...")
        if os.path.exists(drive_db_path):
            shutil.rmtree(drive_db_path) # 기존 백업 삭제
        shutil.copytree(local_db_path, drive_db_path) # 최신 상태 복사

        new_news_to_analyze['vector_db'] = 'Stored'
        print(f"✅ 누적 저장 및 동기화 완료! (총 문서: {vector_db._collection.count()}개)")

        display(new_news_to_analyze[['title', 'source', 'vector_db']].head())
    else:
        print("ℹ️ 추가할 새로운 뉴스 데이터가 없습니다.")
else:
    print("⚠️ 오류: 'new_news_to_analyze' 변수가 정의되지 않았습니다.")

🆕 기존 DB가 없습니다. 새로운 데이터베이스를 생성합니다.
✅ 벡터 DB 연결 성공 (현재 저장된 문서: 0개)
🚀 신규 뉴스 10건 추가 저장을 시작합니다...
💾 업데이트된 데이터를 드라이브로 동기화 중...
✅ 누적 저장 및 동기화 완료! (총 문서: 10개)


,title,source,vector_db
0,I Asked ChatGPT What Bills Retirees Should Eli...,Yahoo_Main,Stored
1,Dell Technologies (DELL): A Key AI Player… But...,Yahoo_Main,Stored
2,Why Analog Devices (ADI) Just Landed On A High...,Yahoo_Main,Stored
3,Texas Instruments (TXN) Is Entering A Critical...,Yahoo_Main,Stored
5,Metals Exploration secures four La India conce...,Yahoo_Main,Stored


In [13]:
# 검색 결과 중 첫 번째 문서의 메타데이터 키(열 이름)들을 출력해봅니다.
if results:
    print("현재 저장된 메타데이터 키 목록:", results[0].metadata.keys())

현재 저장된 메타데이터 키 목록: dict_keys(['title', 'link', 'press', 'date'])


In [14]:
# [수정된 검색 테스트 코드]
query = "최근 국채 금리나 금 가격에 대한 뉴스 있어?"
results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results):
    # 'type' 대신 실제 존재하는 'press'와 'date' (또는 'published_at')를 사용합니다.
    # 키 이름이 press인지 source인지 위 디버깅 코드로 확인 후 수정해 주세요.
    press = doc.metadata.get('press', doc.metadata.get('source', '알 수 없음'))
    date = doc.metadata.get('date', doc.metadata.get('published_at', '날짜 미상'))

    print(f"[{i+1}] {press} ({date}): {doc.page_content[:100]}...")

[1] Yahoo_Main (2026-04-13 09:09:18): [2026-04-13 09:09:18] Yahoo_Main: Metals Exploration secures four La India concessions...
[2] Yahoo_Main (2026-04-13 09:04:00): [2026-04-13 09:04:00] Yahoo_Main: If I Could Tell Investors 1 Thing About the Stock Market Right Now...
[3] Yahoo_Main (2026-04-13 09:08:24): [2026-04-13 09:08:24] Yahoo_Main: Texas Instruments (TXN) Is Entering A Critical Phase – What The $2...


In [15]:
# [Section 03] 지능형 고속 병렬 분석 엔진 - 최종 보완 버전 - 필수 실행
import google.generativeai as genai
import json
import time
import pandas as pd
import os
import concurrent.futures

# 구글 드라이브 데이터 저장 경로 설정
DATA_PATH = "/content/drive/MyDrive/Mecro"

def analyze_sentiment_final(title):
    # ✅ 가장 안정적인 모델명으로 수정했습니다.
    # 'gemini-3.1-flash-lite-preview'를 사용합니다.
    model = genai.GenerativeModel(model_name="models/gemini-3.1-flash-lite-preview")

    prompt = f"""
    당신은 10년 차 글로벌 매크로 전략가입니다. 뉴스 제목을 읽고 시장의 '심리적 온도'를 -1.0에서 1.0 사이로 측정하세요.
    [뉴스 제목]: {title}
    분석 가이드:
    - 매파적(긴축/강달러/성장) 뉘앙스: 0.1 ~ 1.0
    - 비둘기파적(완화/약달러/침체) 뉘앙스: -0.1 ~ -1.0
    - 경제와 무관한 중립 정보: 0.0
    결과는 반드시 아래 JSON 형식으로만 답변하세요:
    {{ "sentiment_score": (수치), "analysis": "한 줄 이유" }}
    """
    try:
        response = model.generate_content(prompt)
        # JSON 문자열 추출 및 정제
        res_text = response.text.strip().replace('```json', '').replace('```', '')
        return json.loads(res_text)
    except Exception:
        return None

def process_row(row):
    result = analyze_sentiment_final(row['title'])
    if result is not None:
        # 데이터프레임 구조에 맞게 반환
        return {
            'Date': row['published_at'],
            'Score': result['sentiment_score'],
            'Title': row['title'],
            'URL': row['url']
        }
    return None

def run_sentiment_pipeline_parallel(all_news_df, max_workers=10, limit_year="2026"):
    if not os.path.exists(DATA_PATH):
        os.makedirs(DATA_PATH)

    SENTIMENT_PATH = f"{DATA_PATH}/sentiment_results.csv"
    columns = ['Date', 'Score', 'Title', 'URL']

    # 1. 기존 데이터 로드 및 초기화
    if os.path.exists(SENTIMENT_PATH):
        try:
            existing_df = pd.read_csv(SENTIMENT_PATH)
            # Date 컬럼이 문자열이므로 limit_year로 필터링
            valid_df = existing_df[existing_df['Date'].astype(str).str.startswith(limit_year)].copy()
            print(f"🧹 {limit_year}년 유효 데이터 {len(valid_df)}건 로드 완료.")
        except:
            valid_df = pd.DataFrame(columns=columns)
    else:
        valid_df = pd.DataFrame(columns=columns)
        print("🆕 새로운 분석 데이터베이스를 생성합니다.")

    # 2. 분석 대상 선정
    if not valid_df.empty:
        to_analyze = all_news_df[~all_news_df['url'].isin(valid_df['URL'])].copy()
    else:
        to_analyze = all_news_df.copy()

    if to_analyze.empty:
        print("✅ 모든 뉴스가 이미 분석되었습니다.")
        return valid_df

    print(f"🚀 신규 뉴스 {len(to_analyze)}건 고속 분석 시작...")

    # 3. 병렬 처리
    new_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_row, row) for _, row in to_analyze.iterrows()]
        for future in concurrent.futures.as_completed(futures):
            res = future.result()
            if res:
                new_results.append(res)

    # 4. 결과 통합 및 저장
    if new_results:
        new_df = pd.DataFrame(new_results)
        final_sentiment_df = pd.concat([valid_df, new_df]).drop_duplicates(subset=['URL'])

        # ✅ 데이터가 있을 때만 정렬 수행 (KeyError 방지)
        if 'Date' in final_sentiment_df.columns:
            final_sentiment_df.sort_values(by='Date', ascending=False, inplace=True)

        final_sentiment_df.to_csv(SENTIMENT_PATH, index=False, encoding='utf-8-sig')
        print(f"✨ 분석 완료! DB 총량: {len(final_sentiment_df)}건")
        return final_sentiment_df
    else:
        print("⚠️ 이번 실행에서 분석된 결과가 없습니다. API 연결 혹은 모델명을 확인해 주세요.")
        return valid_df

# 실행
sentiment_db = run_sentiment_pipeline_parallel(new_news_to_analyze, max_workers=10)

🧹 2026년 유효 데이터 2건 로드 완료.
🚀 신규 뉴스 10건 고속 분석 시작...
✨ 분석 완료! DB 총량: 12건


In [16]:
# [Section 04] 최근 7일 집중 분석 및 RAG 정밀 리포트 - 수정 버전 - 필수 실행
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import google.generativeai as genai

# 1. RAG 기반 인사이트 생성 함수 (기존과 동일)
def get_macro_semiconductor_insight_rag(v_db, m_data, current_index):
    model = genai.GenerativeModel(model_name="models/gemini-3.1-pro-preview")
    query = "반도체 공급망 이슈, AI 서버 수요, 원자재 가격이 반도체 제조에 미치는 영향"
    retrieved_docs = v_db.similarity_search(query, k=10)
    context_text = "\n".join([f"- {doc.page_content}" for doc in retrieved_docs])

    latest_info = ""
    for col in m_data.columns:
        prices = m_data[col].dropna().values
        if len(prices) > 0:
            start_val = prices[0]
            curr_val = prices[-1]
            ret = ((curr_val / start_val) - 1) * 100
            latest_info += f"{col}: {ret:.2f}%, "

    prompt = f"""
    당신은 글로벌 반도체 전문 매크로 투자 전략가입니다. 아래 데이터를 결합하여 리포트를 작성하세요.
    [현재 MarketEcho 지수]: {current_index:.2f}
    [최근 자산 수익률 요약]: {latest_info}
    [뉴스 컨텍스트]: {context_text}
    요청 사항: 뉴스 심리와 원자재 가격 변동이 삼성전자, TSMC 등 반도체 하드웨어 기업에 미치는 구체적 원가 영향과 투자 전략을 제시하세요.
    """
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e: return f"인사이트 추출 실패: {e}"

# 2. ✅ [핵심 수정] 최근 7일 기준 데이터 필터링 로직
sentiment_db['Date'] = pd.to_datetime(sentiment_db['Date'])
market_data.index = pd.to_datetime(market_data.index)

# 마지막 데이터 날짜를 기준으로 7일 전을 시작점으로 설정
plot_end = sentiment_db['Date'].max()
plot_start = plot_end - pd.Timedelta(days=7)

# 필터링 적용 (최근 일주일 데이터만 남김)
focused_db = sentiment_db[(sentiment_db['Date'] >= plot_start) & (sentiment_db['Score'] != 0.0)].copy()
focused_db = focused_db.sort_values('Date')
focused_db['Relative_Index'] = np.cumsum(focused_db['Score'].values)

# market_data도 해당 기간만큼 자름
focused_market = market_data[market_data.index >= plot_start].copy()

# 3. 그래프 생성
fig = make_subplots(specs=[[{"secondary_y": True}]])

# [왼쪽 축] MarketEcho 지수
fig.add_trace(
    go.Scatter(x=focused_db['Date'], y=focused_db['Relative_Index'],
               name='MarketEcho Index', line=dict(color='#1B5E20', width=4)),
    secondary_y=False,
)

# [오른쪽 축] 자산 수익률 (%)
assets_config = {
    "10Y_Bond": {"color": "#D32F2F", "dash": "dot"},
    "Gold": {"color": "#FBC02D", "dash": "solid"},
    "Silver": {"color": "#78909C", "dash": "dash"},
    "Copper": {"color": "#FF9800", "dash": "dashdot"},
    "Aluminum": {"color": "#607D8B", "dash": "dot"}
}

if not focused_market.empty:
    for asset, cfg in assets_config.items():
        if asset in focused_market.columns:
            prices = focused_market[asset].values
            valid_p = prices[~np.isnan(prices)]
            if len(valid_p) > 0:
                # 해당 7일 기간의 시작 가격 기준 수익률 계산
                start_p = valid_p[0]
                returns = ((prices / start_p) - 1) * 100
                fig.add_trace(
                    go.Scatter(x=focused_market.index, y=returns, name=f"{asset} (%)",
                               line=dict(color=cfg['color'], dash=cfg['dash'], width=2)),
                    secondary_y=True,
                )

# 4. 레이아웃 (최근 7일 강조)
fig.update_layout(
    title=f'<b>MarketEcho Focus: Last 7 Days (Ends {plot_end.strftime("%m/%d")})</b>',
    xaxis_title='Timeline (Recent 1 Week)',
    template='plotly_white',
    hovermode='x unified',
    # 축 범위 자동 조절로 1주일 데이터를 꽉 차게 보이게 함
    yaxis=dict(title='Index Momentum', zeroline=True),
    yaxis2=dict(title='Return (%)', side='right', overlaying='y', zeroline=True)
)

fig.show()

# 5. AI 리포트 출력
print("\n📝 [MarketEcho 반도체 매크로 정밀 리포트 - RAG Engine v3]")
print("-" * 60)
current_score = focused_db['Relative_Index'].iloc[-1] if not focused_db.empty else 0
print(get_macro_semiconductor_insight_rag(vector_db, focused_market, current_score))
print("-" * 60)


📝 [MarketEcho 반도체 매크로 정밀 리포트 - RAG Engine v3]
------------------------------------------------------------
**[글로벌 매크로 투자 전략 리포트]**
**제목: AI 인프라 확장의 딜레마 - 원자재 랠리가 반도체 하드웨어 진영(삼성전자/TSMC)에 미치는 원가 압박 및 투자 전략**
**작성일:** 2026년 4월 13일

---

### **1. 매크로 환경 및 자산 시장 동향 요약**
*   **MarketEcho 지수 (2.70):** 시장의 위험 선호 심리(Risk-on)와 AI 주도 성장 기대감은 여전히 강력한 확장 국면에 있습니다.
*   **자산 수익률 분석:** 
    *   **산업용 금속의 폭등:** 구리(Copper) **+5.11%**, 알루미늄(Aluminum) **+2.30%**, 은(Silver) **+2.21%** 
    *   **매크로 지표:** 달러 인덱스(USD) **-1.00%**, 미국 10년물 국채(10Y_Bond) **-0.28%**, 금(Gold) **+0.73%**
    *   **해석:** 약달러 기조 속에서 인플레이션 헷지 및 전력/데이터센터 인프라 구축 수요가 폭발하며 **구리를 필두로 한 산업용 금속 가격이 급등**하고 있습니다. 이는 반도체 및 하드웨어 생태계에 강력한 '원가 상승(Cost Inflation)' 시그널을 보냅니다.

### **2. 뉴스 센티먼트 분석: AI 랠리의 질적 변화**
오늘자 주요 뉴스 흐름은 AI 랠리가 '무차별적 상승'에서 '옥석 가리기' 단계로 진입했음을 시사합니다.
*   **하드웨어/서버 조립 기업의 피로감:** SMCI("단기 전망이 흐려짐"), DELL("지금이 매수 적기인가?") 등 AI 서버 하드웨어 기업들에 대한 경계 심리가 부각되고 있습니다.
*   **장비 및 아날로그 반도체의 부상:** 반면 독점적 장비주인 ASML과 범용/산업용 아날로그 반도체인 TXN, ADI에 

In [15]:
# # [Data Recovery] 잘못 저장된 0.0 데이터 삭제 및 재분석 대상 추출
# import os

# if os.path.exists(SENTIMENT_PATH):
#     full_df = pd.read_csv(SENTIMENT_PATH)
#     # 점수가 0.0인 것은 에러일 확률이 높으므로 제거합니다.
#     cleaned_df = full_df[full_df['Score'] != 0.0].copy()

#     # 0.0이었던 데이터들을 다시 'new_news_to_analyze'로 보내 재분석하게 만듭니다.
#     to_reanalyze_urls = full_df[full_df['Score'] == 0.0]['URL'].tolist()
#     reanalyze_df = news_data[news_data['url'].isin(to_reanalyze_urls)].copy()

#     # 세션에 재분석 대상을 업데이트
#     new_news_to_analyze = reanalyze_df

#     # 기존 파일 업데이트 (0.0이 아닌 것만 남김)
#     cleaned_df.to_csv(SENTIMENT_PATH, index=False, encoding='utf-8-sig')
#     print(f"🧹 청소 완료: {len(full_df) - len(cleaned_df)}건의 에러 데이터를 삭제했습니다.")
#     print(f"🔄 {len(new_news_to_analyze)}건에 대해 다시 분석을 시작할 준비가 되었습니다.")

NameError: name 'SENTIMENT_PATH' is not defined

In [ ]:
# import google.generativeai as genai

# # 1. API 키 설정 (이미 되어 있다면 생략 가능)
# # genai.configure(api_key="GOOGLE_API_KEY")

# print("📡 현재 사용 가능한 모델 리스트를 불러옵니다...\n")

# # 2. 모델 리스트 출력
# for m in genai.list_models():
#     # 텍스트 생성(generateContent)이 가능한 모델만 필터링해서 보기
#     if 'generateContent' in m.supported_generation_methods:
#         print(f"✅ 모델명: {m.name}")
#         print(f"   - 요약: {m.description}")
#         print(f"   - 입력 제한(Tokens): {m.input_token_limit}")
#         print("-" * 50)

# # 3. 임베딩 전용 모델 확인 (Vector DB용)
# print("\n🧬 임베딩 전용 모델 리스트:")
# for m in genai.list_models():
#     if 'embedContent' in m.supported_generation_methods:
#         print(f"✅ 임베딩 모델명: {m.name}")

In [ ]:
# # 삭제할 파일 리스트
# target_files = [
#     "/content/drive/MyDrive/Mecro/data/raw_news.csv",
#     "/content/drive/MyDrive/Mecro/data/sentiment_results.csv",
#     "/content/drive/MyDrive/Mecro/data/market_prices.csv"
# ]

# for file in target_files:
#     if os.path.exists(file):
#         os.remove(file)
#         print(f"✅ 삭제 완료: {file.split('/')[-1]}")

# # 벡터 DB도 함께 삭제
# if os.path.exists(vector_db_path):
#     shutil.rmtree(vector_db_path)
#     print("✅ 삭제 완료: vector_db 폴더")

In [17]:
# 현재 벡터 DB에 저장된 총 데이터 개수 확인
print(f"현재 벡터 DB에 저장된 문서 개수: {vector_db._collection.count()}개")

# 테스트: '반도체'와 관련된 내용이 잘 저장되었는지 검색해보기
query = "반도체 시장 전망"
results = vector_db.similarity_search(query, k=1)

if results:
    print(f"검색 결과 확인: {results[0].page_content[:100]}...")

현재 벡터 DB에 저장된 문서 개수: 10개
검색 결과 확인: [2026-04-13 09:09:06] Yahoo_Main: Why Analog Devices (ADI) Just Landed On A High-Conviction Watchlis...
